## 🎯 Learning Objectives
* Understand why simple accuracy is often insufficient for evaluating machine learning models, especially with imbalanced datasets.
* Define and calculate key classification evaluation metrics: Accuracy, Precision, Recall, and F1-score.
* Interpret a Confusion Matrix and relate its components (True Positives, False Positives, True Negatives, False Negatives) to Precision and Recall.
* Explain the concept of the Receiver Operating Characteristic (ROC) curve and the Area Under the Curve (AUC) as a measure of model performance across various thresholds.
* Identify appropriate evaluation metrics based on the specific business problem and the costs associated with different types of errors.


## ML02-L07: Model Evaluation Metrics: Accuracy, Precision, Recall, F1, AUC

Welcome to a crucial lesson in building robust machine learning models! After training a model, the next critical step is to evaluate its performance. But how do we truly know if our model is 'good'? Simply looking at how many predictions it got 'right' (accuracy) can be misleading, especially in real-world scenarios.

Imagine you're building a model to detect a rare disease that affects only 1% of the population. If your model simply predicts "no disease" for everyone, it would achieve 99% accuracy! While seemingly high, this model is utterly useless because it fails to identify any actual cases. This highlights the limitations of accuracy when dealing with **imbalanced datasets**.

To truly understand our model's strengths and weaknesses, we need a more nuanced set of tools. This lesson will introduce you to the essential evaluation metrics for classification tasks:

1.  **Accuracy:** The proportion of correct predictions out of the total number of predictions. It's a good starting point but can be deceptive with imbalanced classes.

2.  **Confusion Matrix:** A table that summarizes the performance of a classification model. It breaks down predictions into four categories:
    *   **True Positives (TP):** Correctly predicted positive cases.
    *   **True Negatives (TN):** Correctly predicted negative cases.
    *   **False Positives (FP):** Incorrectly predicted positive cases (Type I error, e.g., flagging a healthy person as sick).
    *   **False Negatives (FN):** Incorrectly predicted negative cases (Type II error, e.g., missing a sick person).

3.  **Precision:** Of all the instances predicted as positive, how many were *actually* positive? It answers: "When my model says it's positive, how often is it right?"
    $$Precision = \frac{TP}{TP + FP}$$
    *   **Analogy:** In spam detection, high precision means fewer legitimate emails are incorrectly marked as spam (fewer false alarms).

4.  **Recall (Sensitivity):** Of all the instances that were *actually* positive, how many did the model correctly identify? It answers: "Of all the actual positive cases, how many did my model catch?"
    $$Recall = \frac{TP}{TP + FN}$$
    *   **Analogy:** In disease detection, high recall means fewer actual sick people are missed by the model (fewer false negatives).

5.  **F1-Score:** The harmonic mean of Precision and Recall. It provides a single score that balances both metrics, which is particularly useful when you need a good balance between avoiding false positives and avoiding false negatives.
    $$F1-Score = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

6.  **ROC Curve (Receiver Operating Characteristic) and AUC (Area Under the Curve):**
    *   The **ROC curve** plots the True Positive Rate (Recall) against the False Positive Rate (FP / (FP + TN)) at various classification thresholds. It shows the trade-off between sensitivity and specificity.
    *   The **AUC** is the area under the ROC curve. A higher AUC indicates that the model is better at distinguishing between positive and negative classes across all possible thresholds. An AUC of 0.5 suggests a model performs no better than random guessing, while an AUC of 1.0 indicates a perfect classifier.

Understanding these metrics is crucial for selecting the right model for your specific problem, as the 'best' model often depends on the costs associated with different types of errors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 1. Generate an imbalanced dataset for demonstration
# We'll create a dataset where the positive class is rare (e.g., 5%)
print("Generating an imbalanced dataset...")
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=2,
    n_redundant=10,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=1,
    weights=[0.95, 0.05], # 95% negative class, 5% positive class
    flip_y=0,
    random_state=42
)

print(f"Dataset shape: {X.shape}, Labels shape: {y.shape}")
print(f"Class distribution: 0: {np.sum(y == 0)}, 1: {np.sum(y == 1)}")

# 2. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Train set class distribution: 0: {np.sum(y_train == 0)}, 1: {np.sum(y_train == 1)}")
print(f"Test set class distribution: 0: {np.sum(y_test == 0)}, 1: {np.sum(y_test == 1)}")

# 3. Train a simple Logistic Regression model
print("\nTraining Logistic Regression model...")
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# 4. Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # Probabilities for the positive class

# 5. Calculate and print evaluation metrics
print("\n--- Model Evaluation Metrics ---")

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Display Confusion Matrix visually
plt.figure(figsize=(6, 4))
cmd_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
cmd_display.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.grid(False) # Disable grid for cleaner look
plt.show()

# Precision, Recall, F1-score
# For binary classification, it's often useful to specify pos_label=1 for the positive class
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)

print(f"Precision (for Class 1): {precision:.4f}")
print(f"Recall (for Class 1): {recall:.4f}")
print(f"F1-Score (for Class 1): {f1:.4f}")

# ROC AUC Score
# roc_auc_score requires prediction probabilities, not just binary predictions
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")

# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


### Interpreting the Output and Choosing the Right Metric

Let's break down the output from our code example:

1.  **Dataset Imbalance:** Notice how we deliberately created an imbalanced dataset (95% Class 0, 5% Class 1). This is crucial for understanding why accuracy alone isn't enough.

2.  **Accuracy:** You might see a relatively high accuracy score (e.g., around 0.94-0.95). At first glance, this looks good! However, if our model simply predicted 'Class 0' for everything, it would achieve 95% accuracy. This demonstrates the misleading nature of accuracy with imbalanced data.

3.  **Confusion Matrix:**
    *   The matrix `[[TN, FP], [FN, TP]]` provides the raw counts.
    *   `TN` (top-left): Correctly predicted negative instances.
    *   `FP` (top-right): Incorrectly predicted positive instances (Type I error).
    *   `FN` (bottom-left): Incorrectly predicted negative instances (Type II error).
    *   `TP` (bottom-right): Correctly predicted positive instances.

    In our example, you'll likely see a high `TN` count and a low `TP` count, reflecting the class imbalance. The `FP` and `FN` values are what we really need to scrutinize.

4.  **Precision, Recall, and F1-Score (for Class 1 - the minority class):**
    *   **Precision:** If our model predicted 10 instances as Class 1, and 8 of them were actually Class 1, our precision would be 0.8. A high precision means fewer false alarms. This is critical when false positives are costly (e.g., a bank incorrectly flagging a legitimate transaction as fraud, leading to customer frustration).
    *   **Recall:** If there were 20 actual Class 1 instances, and our model correctly identified 15 of them, our recall would be 0.75. A high recall means fewer missed opportunities. This is critical when false negatives are costly (e.g., a medical test missing a serious disease, or a security system failing to detect an intruder).
    *   **F1-Score:** This metric provides a balance. If you have very high precision but very low recall (or vice-versa), the F1-score will be moderate, indicating that your model is not performing well across both dimensions. It's a good single metric when you need to optimize for both precision and recall simultaneously.

5.  **ROC AUC Score and Curve:**
    *   The **ROC curve** visually represents the trade-off between True Positive Rate (Recall) and False Positive Rate. A curve that hugs the top-left corner indicates a better model.
    *   The **AUC score** quantifies this. An AUC of 0.9 means there's a 90% chance the model will rank a randomly chosen positive instance higher than a randomly chosen negative instance. It's a robust metric because it evaluates the model's performance across *all possible classification thresholds*, making it less sensitive to class imbalance than accuracy.

### Performance Trade-offs and Use Cases

Choosing the right metric depends entirely on the problem's context and the relative costs of different errors:

*   **Prioritize Precision:** When False Positives are expensive or undesirable.
    *   **Examples:** Spam detection (don't want to mark legitimate emails as spam), recommending a product (don't want to recommend irrelevant items), legal document review (don't want to flag irrelevant documents as relevant).

*   **Prioritize Recall:** When False Negatives are expensive or dangerous.
    *   **Examples:** Disease detection (don't want to miss actual cases), fraud detection (don't want to miss actual fraud), security breach detection (don't want to miss an attack).

*   **Prioritize F1-Score:** When you need a good balance between Precision and Recall, especially with imbalanced classes.
    *   **Examples:** Many general classification tasks where both types of errors have significant, but not overwhelmingly different, costs.

*   **Prioritize AUC:** When you need a single metric that summarizes overall model performance across various thresholds and is robust to class imbalance. It's excellent for comparing different models.
    *   **Examples:** Benchmarking different machine learning algorithms, evaluating models in scenarios where the operating point (threshold) might change over time.

In practice, you'll often look at a combination of these metrics to get a complete picture of your model's performance and make informed decisions about its deployment.


### Resources

*   **Scikit-learn Documentation:**
    *   [Metrics and scoring: quantifying the quality of predictions](https://scikit-learn.org/stable/modules/model_evaluation.html)
    *   [sklearn.metrics.accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
    *   [sklearn.metrics.precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
    *   [sklearn.metrics.recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
    *   [sklearn.metrics.f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
    *   [sklearn.metrics.roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
    *   [sklearn.metrics.confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

*   **Google AI Education:**
    *   [Classification: Precision and Recall](https://developers.google.com/machine-learning/crash-course/classification/precision-and-recall)
    *   [Classification: ROC Curve and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)

*   **Further Reading:**
    *   [Understanding AUC - ROC Curve](https://towardsdatascience.com/understanding-auc-roc-curve-e64b00f65b50)
    *   [The F1-Score: A Comprehensive Guide](https://towardsdatascience.com/the-f1-score-a-comprehensive-guide-2026c2bfe12c)
